In [2]:
# Step 1: Required libraries installed
# Step 2: Importing the libraries
import pandas as pd
import os
from keplergl import KeplerGl
from pyproj import CRS
import numpy as np
from matplotlib import pyplot as plt

In [4]:
# Creating a path
path = "/Users/samarjitgehdu/Documents/Achievement 2"

In [6]:
# Creating a file path
file_path = os.path.join(path, "CitiBike_Weather_Merged_2022.csv")

In [8]:
file_path

'/Users/samarjitgehdu/Documents/Achievement 2/CitiBike_Weather_Merged_2022.csv'

In [10]:
# Read the file path


df = pd.read_csv(
    "CitiBike_Weather_Merged_2022.csv",
    dtype={"start_station_id": "string", "end_station_id": "string"},
    index_col = 0)

In [12]:
df.columns

Index(['rideable_type', 'started_at', 'ended_at', 'start_station_name',
       'start_station_id', 'end_station_name', 'end_station_id', 'start_lat',
       'start_lng', 'end_lat', 'end_lng', 'member_casual', 'date', 'avgTemp',
       '_merge'],
      dtype='object')

In [14]:
# Convert 'started_at' to datetime
df['started_at'] = pd.to_datetime(df['started_at'])

In [16]:
# Filter only 2022 rides (To include only the data from 2022)
df = df[df['started_at'].dt.year == 2022]

In [18]:
# Step 3: Data Preprocessing
# Create a value column

df['value'] = 1

In [20]:
df.columns

Index(['rideable_type', 'started_at', 'ended_at', 'start_station_name',
       'start_station_id', 'end_station_name', 'end_station_id', 'start_lat',
       'start_lng', 'end_lat', 'end_lng', 'member_casual', 'date', 'avgTemp',
       '_merge', 'value'],
      dtype='object')

In [22]:
# Step 4: Group trip counts
df_group = df.groupby(['start_station_name', 'end_station_name']).agg({
'value': 'count',
'start_lat': 'first',
'start_lng': 'first',
'end_lat': 'first',
'end_lng': 'first',
    }).reset_index()

In [24]:
# Rename columns
df_group.rename(columns={'value': 'trips'}, inplace=True)

In [26]:
# Verify the dataset
df_group.head()

,start_station_name,end_station_name,trips,start_lat,start_lng,end_lat,end_lng
0,1 Ave & E 110 St,1 Ave & E 110 St,791,40.792337,-73.93824,40.792327,-73.938300
1,1 Ave & E 110 St,1 Ave & E 18 St,2,40.792327,-73.93830,40.733812,-73.980544
2,1 Ave & E 110 St,1 Ave & E 30 St,4,40.792327,-73.93830,40.741444,-73.975361
3,1 Ave & E 110 St,1 Ave & E 39 St,1,40.792327,-73.93830,40.747140,-73.971130
4,1 Ave & E 110 St,1 Ave & E 44 St,12,40.792327,-73.93830,40.750020,-73.969053


In [28]:
# Verify total trips
print(df_group['trips'].sum())

29768282


In [30]:
# Reduce data ONLY for Kepler export (to keep HTML < 100MB)
# Exercise 2.6 just needs an interactive map, not full data.
# ---------------------------------------------------------

# Keep only the top 5,000 most traveled routes
df_group_small = df_group.nlargest(5000, "trips")

In [32]:
print("Reduced dataframe shape:", df_group_small.shape)
df_group_small.head()

Reduced dataframe shape: (5000, 7)


,start_station_name,end_station_name,trips,start_lat,start_lng,end_lat,end_lng
294971,Central Park S & 6 Ave,Central Park S & 6 Ave,12041,40.765909,-73.976342,40.765909,-73.976342
147754,7 Ave & Central Park South,7 Ave & Central Park South,8541,40.766741,-73.979069,40.766741,-73.979069
782289,Roosevelt Island Tramway,Roosevelt Island Tramway,8213,40.757284,-73.953600,40.757284,-73.953600
548187,Grand Army Plaza & Central Park S,Grand Army Plaza & Central Park S,7287,40.764397,-73.973715,40.764397,-73.973715
800503,Soissons Landing,Soissons Landing,7275,40.692317,-74.014866,40.692317,-74.014866


In [34]:
# Step 4: Initialize Kepler.gl Map with REDUCED data
from keplergl import KeplerGl

m = KeplerGl(height=800, data={"data_1": df_group_small})

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


In [36]:
m

KeplerGl(data={'data_1':                        start_station_name                   end_station_name  \
29497…

In [33]:
config = m.config

In [38]:
# Step 5: Export lightweight Kepler HTML for Streamlit dashboard

config = m.config

m.save_to_html(
    file_name="NYC_BikeTrips_2022.html",
    read_only=False,
    config=config
)

print("HTML map exported successfully!")

Map saved to NYC_BikeTrips_2022.html!
HTML map exported successfully!


In [40]:
# Save config as JSON file

import json
with open("kepler_config.json", "w") as f:
    json.dump(config, f)
